# Lancer les entraînements

Ce notebook lance **les expériences qu'on lui nomme**, une à une, dans l'ordre donné. Il n'invente
aucun réglage : tout vient de `configs/experiments/<nom>.yaml`, et chaque run passe par `run_one`,
la fonction que `python -m src.experiments.run` et `make reproduce` appellent l'un comme l'autre.
Ce qu'il ajoute à la ligne de commande : le plan affiché avant le départ, l'état du poste,
l'avancement pendant la campagne, les courbes de perte à l'arrivée, et un noyau où les
enregistrements restent sous la main.

Deux modes, ceux de `make reproduce` :

- `quick` : deux pas d'optimisation sur un corpus tronqué, quelques secondes par expérience. Tout
  ce qu'il écrit va sous `reports/quick/`, ses enregistrements portent `PARTIAL` et n'entrent dans
  aucun tableau.
- `full` : les expériences déclarées, telles quelles, aux destinations réelles. De quelques minutes
  à plusieurs heures selon la sélection.

Un noyau fermé emporte la campagne avec lui. Pour lancer les neuf d'affilée sans garder l'éditeur
ouvert, `make reproduce MODE=full` fait le même travail depuis un terminal, et y survit.

Prérequis : `notebooks/00_environment_check.ipynb` passe, et le corpus est construit
(`make data`).

In [ ]:
from __future__ import annotations

import json
import logging
import os
import sys
from pathlib import Path
from time import perf_counter

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Le repertoire courant, et pas seulement sys.path : une experience lit son
# corpus par le chemin relatif que porte son fichier et ecrit ses checkpoints
# sous runs/. Lance depuis notebooks/, le run creerait notebooks/runs/ et ne
# trouverait pas configs/data/cnn_dailymail.yaml.
os.chdir(REPO_ROOT)

print(f"racine du depot   {REPO_ROOT}")
print(f"interpreteur      {sys.executable}")

## 1. Les expériences à lancer

**Ce carnet présente des résultats, il n'entraîne pas.** `ENTRAINER` vaut `False` : un Run All
relit les enregistrements déjà écrits sous `reports/results/` et n'occupe pas le GPU. La campagne
se lance au terminal, `make reproduce MODE=full`, qui enchaîne en plus les ablations, les tableaux
du rapport et les figures. Passer `ENTRAINER` à `True`
fait entraîner la section 4 depuis ici, au prix d'un noyau qui doit rester vivant du début à la
fin : fermer VS Code emporte la campagne.

`EXPERIMENTS` nomme les expériences décrites, dans l'ordre donné. `None` prend les neuf déclarées
sous `configs/experiments/`, dans l'ordre de la campagne : le Transformer from scratch d'abord,
des plus petites proportions aux plus grandes, puis `t5-small` zero-shot, puis ses fine-tunes.
C'est l'ordre de `make reproduce` : le modèle from scratch est ce à quoi le reste se compare, et
une campagne interrompue a alors mesuré une famille entière plutôt que trois fine-tunes sans
référence. Une liste plus courte restreint à ce qu'on veut lire, ou à ce qui reste
à lancer quand `ENTRAINER` est `True`.

Les réglages suivants disent où écrire, pas comment entraîner. Un hyperparamètre se change dans le
fichier YAML, où il est versionné, jamais ici. Ils ne servent que si `ENTRAINER` est `True`.

`EVALUATION_LIMIT` a une conséquence enregistrée : évaluer les N premiers documents fait porter au
run le statut `PARTIAL`, et un run partiel n'entre dans aucun tableau. Un score mesuré sur
cinquante documents et un score mesuré sur le jeu de test complet ne se comparent pas.

`RESUME` reprend l'entraînement depuis les checkpoints déjà présents sous `runs/<nom>/`. Utile
après une interruption, dangereux après un changement de configuration : les poids repris viennent
de l'ancienne, et le run écrira pourtant la nouvelle dans son enregistrement. Sur une sélection de
plusieurs expériences, il s'applique à toutes.

`FOCUS` ne lance rien. Il nomme l'expérience que les sections 5 à 8 détaillent ; `None` prend la
dernière qui a rendu une mesure.


In [ ]:
#: Ce carnet presente des resultats ; il n'entraine pas. Un Run All relit les
#: enregistrements deja sur le disque et n'occupe pas le GPU. La campagne se
#: lance au terminal, par make reproduce MODE=full, qui enchaine aussi les
#: ablations, les tableaux et les figures que ce carnet ne fait pas.
#:
#: True fait entrainer la section 4 depuis ici. Le noyau doit alors rester
#: vivant du debut a la fin : fermer VS Code emporte la campagne.
ENTRAINER = False

from src.experiments.config import (
    DEFAULT_EXPERIMENTS_DIR,
    ExperimentConfig,
    campaign_order,
    iter_experiment_files,
    load_experiment_config,
)
from src.experiments.record import STATUS_NOT_RUN, find_record
from src.experiments.run import DEFAULT_RESULTS_DIR, DEFAULT_RUNS_DIR

#: Experiences lancees, dans cet ordre. None prend les neuf declarees.
EXPERIMENTS: list[str] | None = None

#: quick verifie la chaine et ne produit aucun resultat, full joue les
#: experiences declarees. Meme vocabulaire que make reproduce MODE=...
MODE = "full"

#: Reprend l'entrainement depuis les checkpoints du repertoire de run.
RESUME = False

#: Envoi des runs termines vers MLflow. Un run quick n'est jamais trace.
TRACKING = True

#: Evalue les N premiers documents du split seulement, et marque le run
#: PARTIAL. None evalue le split entier.
EVALUATION_LIMIT: int | None = None

#: Experience que les sections 5 a 8 detaillent. None prend la derniere qui
#: a rendu une mesure.
FOCUS: str | None = None

# Les fichiers se lisent en ordre de nom, la campagne se joue dans un autre :
# le from scratch d'abord, puis le zero-shot, puis les fine-tunes. C'est
# l'ordre de make reproduce, et celui qui laisse une famille entiere mesuree
# quand un Run All s'arrete au milieu.
DECLARED: list[ExperimentConfig] = []
FILES: dict[str, Path] = {}
for path in iter_experiment_files(DEFAULT_EXPERIMENTS_DIR):
    loaded = load_experiment_config(path)
    DECLARED.append(loaded)
    FILES[loaded.name] = path
CONFIGS: dict[str, ExperimentConfig] = {
    config.name: config for config in campaign_order(DECLARED)
}

SELECTION = list(CONFIGS) if EXPERIMENTS is None else list(EXPERIMENTS)
unknown = [name for name in SELECTION if name not in CONFIGS]
if unknown:
    raise SystemExit(
        f"{', '.join(unknown)} : non declare(s) sous {DEFAULT_EXPERIMENTS_DIR}. "
        f"Noms disponibles : {', '.join(CONFIGS)}"
    )
if not SELECTION:
    raise SystemExit("EXPERIMENTS est vide : il n'y a rien a lancer.")
if FOCUS is not None and FOCUS not in SELECTION:
    raise SystemExit(f"FOCUS vaut {FOCUS!r}, qui n'est pas dans la selection.")

print(f"{'experience':<24} {'variante':<22} {'corpus':<12} enregistrement")
for name, config in CONFIGS.items():
    found = find_record(DEFAULT_RESULTS_DIR, name)
    percentage = config.dataset.percentage
    proportion = "sans objet" if percentage is None else f"{percentage} %"
    marker = ">" if name in SELECTION else " "
    print(
        f"{marker} {name:<22} {config.variant:<22} {proportion:<12} "
        f"{STATUS_NOT_RUN if found is None else found.status}"
    )
print()
print(f"{len(SELECTION)} selectionnee(s), dans l'ordre : {', '.join(SELECTION)}")

## 2. Le poste et le corpus

Seules deux choses peuvent empêcher ce notebook d'aller au bout ; le contrôle complet de
l'environnement est le travail de `00_environment_check.ipynb`.

Le matériel est lu par `describe_hardware()`, la fonction que chaque run enregistre dans MLflow,
de sorte que la durée observée plus bas reste interprétable à côté de celles de `reports/`. Un
poste sans GPU n'est pas bloqué : il est lent, et la précision mixte n'y sert à rien.

Le corpus est bloquant. Sa version est ce qui rend deux runs comparables : un score obtenu sur un
autre `dataset_version` ne se compare à rien de ce que le rapport publie. Absent, il arrête le
notebook plutôt que de laisser un entraînement échouer trois cellules plus loin.

In [ ]:
from src.data.config import load_pipeline_config
from src.data.dataset import MANIFEST_NAME, load_manifest
from src.utils.device import describe_hardware, resolve_device

for key, value in describe_hardware().items():
    print(f"{key:<18} {value}")

# Une selection dont les experiences ne demandent pas le meme peripherique ne
# se lit pas comme une campagne : la duree d'un run CPU et celle d'un run GPU
# ne se comparent pas.
requested = {
    (CONFIGS[name].training.device if CONFIGS[name].training else "auto") for name in SELECTION
}
if len(requested) > 1:
    raise SystemExit(f"la selection demande plusieurs peripheriques : {sorted(requested)}")
device = resolve_device(next(iter(requested)))
print(f"{'peripherique':<18} {device} (demande : {next(iter(requested))})")
print()

# Meme raison pour le corpus : deux corpus dans une selection, ce sont deux
# campagnes, et leurs scores ne se rangent pas dans le meme tableau.
declared = {CONFIGS[name].dataset.config for name in SELECTION}
if len(declared) > 1:
    raise SystemExit(f"la selection melange {len(declared)} corpus : {sorted(map(str, declared))}")

pipeline = load_pipeline_config(next(iter(declared)))
processed = pipeline.paths.processed
if not (processed / MANIFEST_NAME).is_file():
    raise SystemExit(f"corpus absent de {processed}. Le construire : make data")

manifest = load_manifest(processed)
print(f"{'corpus':<18} {manifest.name} (source : {manifest.source_dataset})")
print(f"{'version':<18} {manifest.dataset_version[:16]}...")
print(f"{'tokenizer':<18} {manifest.tokenizer}")
for split, size in manifest.splits.items():
    print(f"{'split ' + split:<18} {size} exemples")

## 3. Le plan de la campagne

Ce qui est imprimé est ce que la boucle va exécuter, ligne par ligne. La colonne `écrase` dit ce
qui se trouve déjà sur le disque sous ce nom : le run l'écrasera, car un score que le code actuel
ne sait plus produire n'est plus un résultat.

Une sélection d'une seule expérience imprime en plus sa configuration champ par champ.

En mode `quick`, `quick_experiment` remplace le budget par celui d'une répétition : une époque,
deux pas, un décodage court. Il ne touche ni à la graine, ni au split, ni à l'architecture, ni à la
proportion déclarée : une vérification qui changerait ces valeurs vérifierait une autre expérience.
`max_steps` est ce qui rend le run `PARTIAL`, et les destinations basculent sous `reports/quick/` et
`runs/quick/`.

In [ ]:
from src.experiments.record import STATUS_OK, STATUS_PARTIAL, run_directory
from src.tracking.store import describe_store
from src.experiments.reproduce import (
    MODE_QUICK,
    MODES,
    QUICK_DESTINATIONS,
    QUICK_EVALUATION_EXAMPLES,
    quick_experiment,
)

if MODE not in MODES:
    raise SystemExit(f"MODE doit valoir l'un de {MODES}, pas {MODE!r}.")

quick = MODE == MODE_QUICK
RESULTS_DIR = QUICK_DESTINATIONS.results if quick else DEFAULT_RESULTS_DIR
RUNS_DIR = QUICK_DESTINATIONS.runs if quick else DEFAULT_RUNS_DIR
LIMIT = QUICK_EVALUATION_EXAMPLES if quick else EVALUATION_LIMIT
TRACES = TRACKING and not quick

#: Ce que la boucle va executer, dans l'ordre de SELECTION.
PLANNED: dict[str, ExperimentConfig] = {
    name: quick_experiment(CONFIGS[name]) if quick else CONFIGS[name] for name in SELECTION
}

print(f"mode              {MODE}")
print(f"resultats         {RESULTS_DIR}")
print(f"checkpoints       {RUNS_DIR}")
print(f"evaluation        {'split entier' if LIMIT is None else f'{LIMIT} premiers documents'}")
print(f"reprise           {'oui, depuis les checkpoints presents' if RESUME else 'non'}")
print(f"tracage MLflow    {'oui' if TRACES else 'non'}")
if TRACES:
    # Le magasin est affiche avant le depart, mot de passe masque : une
    # campagne qui aurait trace dans un fichier local au lieu de la base
    # partagee se voit ici et non apres coup.
    print(f"magasin           {describe_store()}")
print()

print(f"  {'experience':<22} {'corpus':<20} {'epoques':<11} {'attendu':<9} ecrase")
for name, config in PLANNED.items():
    partial = LIMIT is not None or config.capped
    previous = find_record(RESULTS_DIR, name)
    percentage = config.dataset.percentage
    if percentage is None:
        share = "sans objet"
    else:
        examples = manifest.splits["train"] * percentage // 100
        share = f"{percentage} % ({examples} ex.)"
    epochs = "sans objet" if config.training is None else str(config.training.epochs)
    expected = STATUS_PARTIAL if partial else STATUS_OK
    print(
        f"  {name:<22} {share:<20} {epochs:<11} {expected:<9} "
        f"{'-' if previous is None else previous.status}"
    )

if len(PLANNED) == 1:
    # Une selection d'une seule experience garde le detail champ par champ :
    # c'est ce qu'on relit avant de lancer un run isole.
    only = next(iter(PLANNED.values()))
    print()
    print(f"{only.name} : {only.experiment.description}")
    print(f"variante          {only.variant}")
    print(f"fichier           {FILES[only.name]}")
    print(f"graine            {only.experiment.seed}")
    print()
    for label, block in (
        ("modele", only.model),
        ("entrainement", only.training),
        ("evaluation", only.evaluation),
    ):
        if block is None:
            print(f"{label} : aucun bloc, ce run n'entraine pas")
            print()
            continue
        print(f"{label} :")
        for key, value in block.model_dump().items():
            print(f"  {key:<28} {value}")
        print()

## 4. La campagne

**Par défaut, cette cellule lit.** `ENTRAINER` valant `False`, elle relit les enregistrements de
`reports/results/` et n'appelle jamais `run_one` : rien n'est entraîné, rien n'est écrasé, le GPU
reste libre. Le tableau final est le même que celui d'une campagne, avec `ABSENT` pour les
expériences qui n'ont pas encore été produites. C'est le mode à utiliser après un
`make reproduce MODE=full` au terminal.

Ce qui suit ne vaut que pour `ENTRAINER = True`.

La cellule configure le journal avant de lancer quoi que ce soit : le `LoggingCallback` publie
l'avancement sur le logger `syntra.training`, en `INFO`. Sans racine configurée, un entraînement
d'une heure n'afficherait rien entre sa bannière et son résumé final. `force=True` parce qu'un
noyau qui a déjà exécuté autre chose peut porter des gestionnaires, et `basicConfig` ne ferait
alors rien.

**Suivre.** Chaque expérience ouvre une bannière `i/N`, puis ses pas et ses époques s'écrivent au
fil de l'eau. Une ligne de bilan la referme : statut, durée, position dans la campagne et temps
cumulé. Aucun temps restant n'est estimé : les neuf expériences n'ont ni le même corpus ni le même
budget d'époques, et `pretrained_zero_shot` n'entraîne pas du tout ; une moyenne des runs déjà
faits annoncerait n'importe quoi. `make mlflow-ui` sert la même campagne dans un navigateur, run
par run, pendant qu'elle tourne.

`run_one` **ne lève pas**. Une expérience qui casse est écrite avec le statut `FAILED` et son
message d'erreur, puis rendue comme les autres, et la campagne continue sur la suivante. Une
cellule qui se termine sans traceback n'est donc pas une cellule où tout a réussi : c'est le
tableau de statuts qui le dit.

Interrompre le noyau arrête la campagne à l'expérience en cours. Ce qui a déjà été rendu reste
dans `RECORDS` et sur le disque, et le bilan s'imprime quand même. Les checkpoints de l'expérience
interrompue restent sous `runs/<nom>/`, et `RESUME = True` repart de là ; aucun enregistrement
n'est produit pour elle en revanche.

Ordre de grandeur sur le poste de référence du dépôt (RTX 5060 portable) : environ trois heures
pour la campagne complète des neuf expériences, quelques secondes pour une répétition `quick`.


In [ ]:
from src.experiments.record import RunRecord, find_record
from src.experiments.reproduce import quick_corpus
from src.experiments.run import format_record, load_corpus, run_one
from src.metrics.rouge import REPORTED_VARIANT
from src.tracking.client import DEFAULT_EXPERIMENT, build_tracker
from src.utils.device import release_accelerator

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s: %(message)s",
    stream=sys.stdout,
    force=True,
)


def duration(seconds: float) -> str:
    # Une campagne se lit en heures, un run quick en secondes. La meme unite
    # pour les deux rendrait l'un des deux illisible.
    if seconds < 120:
        return f"{seconds:.1f} s"
    if seconds < 7200:
        return f"{seconds / 60:.1f} min"
    return f"{seconds / 3600:.2f} h"


#: Les enregistrements rendus, dans l'ordre de passage. Une interruption
#: laisse ici ce qui avait deja tourne.
RECORDS: dict[str, RunRecord] = {}
interrupted: str | None = None
campaign = perf_counter()

if not ENTRAINER:
    # Lecture seule. C'est le defaut : le carnet decrit une campagne, il ne la
    # produit pas. Rien n'est ecrase, aucun GPU n'est pris.
    print(f"ENTRAINER est False : lecture de {RESULTS_DIR}, aucun entrainement lance.")
    for name in PLANNED:
        found = find_record(RESULTS_DIR, name)
        if found is not None:
            RECORDS[name] = found
else:
    tracker = build_tracker(enabled=TRACES, tracking_uri=None, experiment=DEFAULT_EXPERIMENT)
    for position, (name, planned) in enumerate(PLANNED.items(), start=1):
        print()
        print(f"=== {position}/{len(PLANNED)}  {name} ({planned.variant}) ===")
        started = perf_counter()
        try:
            # Le corpus tronque est passe au run plutot que reconstruit : le mode
            # quick doit s'entrainer sur une poignee d'exemples sans toucher au
            # corpus fige.
            RECORDS[name] = run_one(
                planned,
                results_dir=RESULTS_DIR,
                runs_dir=RUNS_DIR,
                limit=LIMIT,
                resume=RESUME,
                tracker=tracker,
                corpus=quick_corpus(load_corpus(planned)) if quick else None,
            )
        except KeyboardInterrupt:
            interrupted = name
            print()
            print(f"{name} interrompu apres {duration(perf_counter() - started)}, campagne arretee.")
            break
        # Meme raison que dans make reproduce : torch garde les blocs liberes
        # dans son allocateur, et ce que les experiences precedentes ont
        # reserve est encore tenu quand la suivante construit son modele.
        release_accelerator()
        print()
        print(
            f"{name} rendu en {duration(perf_counter() - started)}, "
            f"statut {RECORDS[name].status}"
            f"  [{position}/{len(PLANNED)}, {duration(perf_counter() - campaign)} cumulees]"
        )

print()
print("=" * 78)
print(f"  {'experience':<22} {'statut':<12} {'duree':<10} {REPORTED_VARIANT}")
for name in PLANNED:
    record = RECORDS.get(name)
    if record is None:
        if not ENTRAINER:
            absent = "ABSENT"
        else:
            absent = "INTERROMPU" if name == interrupted else "NON LANCE"
        print(f"  {name:<22} {absent:<12}")
        continue
    score = record.rouge(REPORTED_VARIANT)
    print(
        f"  {name:<22} {record.status:<12} {duration(record.duration_seconds):<10} "
        f"{'-' if score is None else format(score, '.4f')}"
    )
print()
if ENTRAINER:
    print(f"{len(RECORDS)}/{len(PLANNED)} rendues en {duration(perf_counter() - campaign)}.")
else:
    print(f"{len(RECORDS)}/{len(PLANNED)} relues depuis le disque.")
    if len(RECORDS) < len(PLANNED):
        print("Les manquantes se produisent au terminal : make reproduce MODE=full")

# Les sections suivantes decrivent une seule experience. FOCUS la nomme ; sans
# FOCUS, c'est la derniere qui a rendu une mesure, faute de quoi la derniere
# rendue tout court.
measured = [name for name, record in RECORDS.items() if record.measured]
CHOSEN = FOCUS or (measured[-1] if measured else (list(RECORDS)[-1] if RECORDS else None))
if CHOSEN is None:
    raise SystemExit(
        f"aucun enregistrement sous {RESULTS_DIR} : les sections 5 a 8 n'ont rien a decrire. "
        "Lancer la campagne au terminal (make reproduce MODE=full), ou passer ENTRAINER a True."
    )
RECORD = RECORDS[CHOSEN]
CONFIG = PLANNED[CHOSEN]
DIRECTORY = run_directory(RESULTS_DIR, CHOSEN, partial=RECORD.status == STATUS_PARTIAL)
print(f"sections 5 a 8   {CHOSEN}")


## 5. Ce que le run choisi a produit

Les quatre sections qui suivent décrivent **une** expérience, celle que la section 4 a retenue sous
`CHOSEN`. Pour en lire une autre, nommer `FOCUS` en section 1 et réexécuter à partir de la
section 4 : rien n'est réentraîné, les enregistrements sont relus sur le disque.

`format_record` est la fonction appelée en fin de ligne de commande : ce bloc est mot pour mot ce
qu'aurait affiché `python -m src.experiments.run`. Les scores viennent avec leur intervalle
bootstrap, et un run partiel le dit lui-même.

Les fichiers listés ensuite sont ce qui reste du run sur le disque : l'enregistrement que les
tableaux relisent, les courbes que les figures tracent, les prédictions produites, et les
checkpoints.


In [ ]:
from src.experiments.record import STATUS_FAILED

print(format_record(RECORD))

if RECORD.status == STATUS_FAILED:
    # Un echec est enregistre dans le repertoire nominal : il n'y a pas de
    # mesure partielle a isoler, il n'y a pas de mesure du tout.
    DIRECTORY = run_directory(RESULTS_DIR, RECORD.experiment)

print()
print(f"repertoire du run {DIRECTORY}")
for produced in sorted(DIRECTORY.glob("*")) if DIRECTORY.is_dir() else []:
    print(f"  {produced.name:<24} {produced.stat().st_size / 1024:.1f} Kio")

checkpoints = Path(RUNS_DIR) / RECORD.experiment
if checkpoints.is_dir():
    print(f"checkpoints       {checkpoints}")
    for weight in sorted(checkpoints.glob("*.pt")):
        print(f"  {weight.name:<24} {weight.stat().st_size / 1048576:.1f} Mio")

## 6. Les courbes

Elles sont lues dans `history.json`, écrit par le `HistoryCallback` pendant le run : ces listes
sont celles que `make figures` trace pour le rapport, à la mise en forme près.

`matplotlib.pyplot` est importé directement, sans passer par `src.experiments.figures.pyplot()`,
qui fixe le backend `Agg` pour dessiner sans écran. Ici l'écran existe, et forcer `Agg`
reviendrait à ne rien afficher.

La perte par pas montre si l'optimisation avance ; la perte par époque montre où l'écart entre
entraînement et validation se creuse, donc à partir de quand l'arrêt anticipé avait raison. Une
courbe de deux points, en mode `quick`, ne montre ni l'un ni l'autre.

In [ ]:
import matplotlib.pyplot as plt

from src.experiments.record import HISTORY_FILE

history_path = DIRECTORY / HISTORY_FILE
if not history_path.is_file():
    print(f"aucune courbe : {history_path} est absent, ce run n'a pas entraine.")
else:
    curves = json.loads(history_path.read_text(encoding="utf-8"))["curves"]
    step_loss = curves["step_loss"]
    train_loss = curves["train_loss"]
    validation_loss = curves["validation_loss"]

    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(range(1, len(step_loss) + 1), step_loss, linewidth=0.8)
    axes[0].set_title("perte par pas d'optimisation")
    axes[0].set_xlabel("pas")
    axes[0].set_ylabel("perte")

    epochs = range(1, len(train_loss) + 1)
    axes[1].plot(epochs, train_loss, marker="o", label="entrainement")
    axes[1].plot(epochs, validation_loss, marker="o", label="validation")
    axes[1].set_title("perte par epoque")
    axes[1].set_xlabel("epoque")
    axes[1].legend()

    for axis in axes:
        axis.grid(alpha=0.3)
    figure.suptitle(f"{RECORD.experiment} ({RECORD.status})")
    figure.tight_layout()
    plt.show()

## 7. Les scores et leurs intervalles

Trois barres, et l'intervalle qui va avec chacune. C'est l'intervalle qui décide de ce qu'on a le
droit de lire : deux scores dont les intervalles se recouvrent ne se classent pas.

Un run `PARTIAL` produit un score sur une poignée de documents. Il est tracé pour vérifier que la
chaîne d'évaluation fonctionne, jamais pour être lu comme une mesure.

In [ ]:
from src.metrics.rouge import ROUGE_VARIANTS

report = (RECORD.evaluation or {}).get("report", {})
rouge = report.get("rouge", {})

if not rouge:
    print("aucun score : ce run n'a pas ete evalue.")
else:
    scores = rouge["scores"]
    intervals = rouge.get("intervals", {})

    values = [scores[v]["fmeasure"] for v in ROUGE_VARIANTS]
    lows = [values[i] - intervals.get(v, {}).get("low", values[i]) for i, v in enumerate(ROUGE_VARIANTS)]
    highs = [intervals.get(v, {}).get("high", values[i]) - values[i] for i, v in enumerate(ROUGE_VARIANTS)]

    figure, axis = plt.subplots(figsize=(7, 4))
    bars = axis.bar(ROUGE_VARIANTS, values, color=["#4c72b0", "#dd8452", "#55a868"], width=0.55)
    axis.errorbar(ROUGE_VARIANTS, values, yerr=[lows, highs], fmt="none", ecolor="#333", capsize=6)
    for bar, value in zip(bars, values, strict=True):
        axis.text(bar.get_x() + bar.get_width() / 2, value, f"{value:.4f}",
                  ha="center", va="bottom", fontsize=10)
    axis.set_ylabel("F-mesure")
    axis.set_ylim(0, max(values) * 1.35)
    axis.grid(axis="y", alpha=0.3)
    figure.suptitle(f"{RECORD.experiment} ({RECORD.status}) sur {report.get('size', '?')} documents")
    figure.tight_layout()
    plt.show()

    print(f"{'longueur moyenne':<22} {report['prediction_words']['mean']:.1f} mots produits, "
          f"{report['reference_words']['mean']:.1f} en reference")
    print(f"{'predictions vides':<22} {report['empty_predictions']}")

## 8. Ce que le magasin a reçu

Le run vient d'être écrit sur disque, puis miroité dans MLflow : métadonnées et métriques dans
PostgreSQL, artefacts et poids dans `MLFLOW_ARTIFACT_ROOT`. La cellule suivante relit le magasin
plutôt que de faire confiance au fait que l'appel n'a pas levé.

Le modèle déposé est celui qui a été **évalué**, relu depuis le meilleur checkpoint. Quand l'arrêt
anticipé a retenu une époque antérieure, ce n'est pas l'objet en fin d'entraînement. Seul un run
`OK` entre au registre : un `PARTIAL` a vu deux pas d'optimisation, ses poids ne veulent rien
dire.

`make mlflow-ui` sert la même chose dans un navigateur, sur <http://localhost:5000>.

In [ ]:
from src.tracking.model import registered_name

if not TRACES:
    print("run non trace : rien a lire dans le magasin.")
    print("Mettre TRACKING = True et MODE = 'full' pour y deposer un run.")
else:
    import mlflow

    from src.metrics.rouge import REPORTED_VARIANT
    from src.tracking.client import DEFAULT_EXPERIMENT
    from src.tracking.payload import TAG_PREFIX
    from src.tracking.store import resolve_artifact_root, resolve_tracking_uri

    mlflow.set_tracking_uri(resolve_tracking_uri())
    client = mlflow.MlflowClient()
    print(describe_store())
    print()

    experiment = client.get_experiment_by_name(DEFAULT_EXPERIMENT)
    found = client.search_runs(
        [experiment.experiment_id],
        filter_string=f"attributes.run_name = '{RECORD.experiment}'",
        order_by=["attributes.start_time DESC"],
        max_results=1,
    )
    if not found:
        print(f"aucun run nomme {RECORD.experiment!r} dans {DEFAULT_EXPERIMENT!r}.")
    else:
        run = found[0]
        # Les tags portent le prefixe du projet : sur une base partagee, un tag
        # nu entrerait en collision avec ceux d'une autre application.
        tag = lambda key: run.data.tags.get(f"{TAG_PREFIX}.{key}", "inconnu")
        print(f"{'run':<24} {run.info.run_name}  ({run.info.run_id})")
        print(f"{'statut':<24} {run.info.status}, enregistrement {tag('status')}")
        print(f"{'commit':<24} {tag('git_commit')[:12]} sur {tag('git_branch')}"
              f", arbre modifie : {tag('git_dirty')}")
        print(f"{'materiel':<24} {tag('hardware.gpu_name')}, torch {tag('hardware.torch_version')}")
        print(f"{'corpus':<24} {run.data.params.get('dataset.name', '?')}"
              f" {run.data.params.get('dataset.version', '?')[:16]}")
        print()
        print("scores traces :")
        for variant in ("rouge1", "rouge2", REPORTED_VARIANT):
            f = run.data.metrics.get(f"{variant}_f")
            low = run.data.metrics.get(f"{variant}_low")
            high = run.data.metrics.get(f"{variant}_high")
            if f is not None:
                print(f"  {variant:<10} {f:.4f}  [{low:.4f}, {high:.4f}]")

    name = registered_name(RECORD.experiment)
    versions = client.search_model_versions(f"name='{name}'")
    print()
    if not versions:
        print(f"registre : aucune version sous {name!r}.")
        print("Attendu pour un run PARTIAL ou FAILED, qui n'est pas une mesure.")
    else:
        latest = max(versions, key=lambda v: int(v.version))
        flavour = "transformers" if CONFIG.variant.startswith("pretrained") else "pytorch"
        print(f"registre : {latest.name} version {latest.version}, statut {latest.status}")
        print(f"recharger :  mlflow.{flavour}.load_model('models:/{latest.name}/{latest.version}')")

    root = Path(resolve_artifact_root())
    if root.is_dir():
        total = sum(f.stat().st_size for f in root.rglob("*") if f.is_file())
        print(f"artefacts : {root} occupe {total / 1048576:.1f} Mio")

## 9. La suite

Un run terminé ne met pas le rapport à jour, et ce notebook ne le déclenche pas tout seul. C'est
ce qui fait de `make report-sync` une cible distincte de l'entraînement : régénérer les tableaux
depuis un poste où deux expériences sur neuf ont tourné réécrirait les sept autres en `NOT_RUN`.

La cellule ne regarde donc pas ce que la campagne vient de rendre, mais ce que `reports/results/`
porte pour les neuf expériences déclarées, tous runs confondus. C'est là que `report-sync` lit,
quel que soit le mode de ce notebook : une répétition `quick` n'y écrit rien et ne change donc
aucune de ces lignes.

In [ ]:
declared_status = {name: find_record(DEFAULT_RESULTS_DIR, name) for name in CONFIGS}
complete = [name for name, record in declared_status.items()
            if record is not None and record.status == STATUS_OK]
missing = [name for name in CONFIGS if name not in complete]

print(f"dans {DEFAULT_RESULTS_DIR} : {len(complete)}/{len(CONFIGS)} experiences en {STATUS_OK}")
print()
print("Depuis un terminal, a la racine du depot :")
print()
if not missing:
    print("  make report-sync")
    print("  make figures")
else:
    print(f"  report-sync reecrirait {len(missing)} ligne(s) en NOT_RUN :")
    for name in missing:
        found = declared_status[name]
        print(f"    {name:<24} {'-' if found is None else found.status}")
    print("  les lancer d'abord, ici ou avec make reproduce MODE=full")
print("  make mlflow-ui        # http://localhost:5000")